
# 01 - Data Cleaning Notebook
## Workplace Safety Data Analysis

**Objetivo:** Identificar y corregir problemas de calidad de datos para construir una versión confiable del dataset que será utilizada en el análisis exploratorio y la generación de KPIs.

### Problemas esperados
- Valores nulos
- Duplicados
- Inconsistencias de texto
- Outliers
- Fechas inválidas o futuras
- Registros fuera de reglas de negocio


In [22]:

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

file_path = 'data/raw/workplace_safety_dataset_raw_with_quality_issues.csv'

df = pd.read_csv(file_path)

print(f'Registros: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

df.head()

Registros: 775
Columnas: 17


,employee_id,age,gender,department,job_role,years_experience,shift,accident_date,accident_type,severity,lost_days,ppe_used,training_completed,fatigue_level,root_cause,city,overtime_hours
0,1271,60,female,Operations,Operator,19.5,Day Shift,2023-04-13,Cut,Moderate,2,Yes,Yes,8.0,Fatigue,Bogotá,11.0
1,1131,39,Female,Quality,Operator,1.4,Day Shift,2024-10-25,Ergonomic,Moderate,8,Yes,Yes,3.0,Human Error,Barranquilla,8.6
2,1055,20,male,Production,Operator,21.5,Day,2025-05-25,Fall,Moderate,6,No,NaN,10.0,Fatigue,Bogotá,11.8
3,1053,19,male,Warehouse,Operator,17.1,night shift,2025-02-15,Slip,Minor,7,Yes,No,4.0,Human Error,Cali,3.5
4,1002,23,Female,Production,Coordinator,5.2,day,2024-07-15,Fall,Moderate,0,No,Yes,8.0,Fatigue,Cali,3.6


In [23]:
print(df.shape)


(775, 17)


## 1. Perfilamiento Inicial de Datos

In [24]:

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 775 entries, 0 to 774
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   employee_id         775 non-null    int64  
 1   age                 775 non-null    int64  
 2   gender              775 non-null    object 
 3   department          775 non-null    object 
 4   job_role            775 non-null    object 
 5   years_experience    775 non-null    float64
 6   shift               775 non-null    object 
 7   accident_date       775 non-null    object 
 8   accident_type       775 non-null    object 
 9   severity            775 non-null    object 
 10  lost_days           775 non-null    int64  
 11  ppe_used            775 non-null    object 
 12  training_completed  735 non-null    object 
 13  fatigue_level       735 non-null    float64
 14  root_cause          775 non-null    object 
 15  city                775 non-null    object 
 16  overtime

In [26]:

df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
employee_id,775.0,NaN,NaN,NaN,1153.340645,86.199906,1001.0,1077.5,1156.0,1230.0,1299.0
age,775.0,NaN,NaN,NaN,38.934194,13.254498,12.0,28.0,38.0,51.0,120.0
gender,775,7,F,141,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department,775,6,Quality,156,NaN,NaN,NaN,NaN,NaN,NaN,NaN
job_role,775,5,Operator,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_experience,775.0,NaN,NaN,NaN,13.100129,7.182979,0.0,6.8,13.4,19.15,25.0
shift,775,7,day,144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
accident_date,775,509,2025-05-30,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
accident_type,775,6,Struck By Object,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
severity,775,3,Minor,435,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Identificación de Valores Nulos

In [28]:

nulls = df.isnull().sum().sort_values(ascending=False)
nulls[nulls > 0]


overtime_hours        42
fatigue_level         40
training_completed    40
dtype: int64

### Tratamiento de valores nulos

In [29]:

df['fatigue_level'] = df['fatigue_level'].fillna(df['fatigue_level'].median())
df['overtime_hours'] = df['overtime_hours'].fillna(df['overtime_hours'].median())
df['training_completed'] = df['training_completed'].fillna('Unknown')


## 3. Detección y Eliminación de Duplicados

In [30]:

duplicates = df.duplicated().sum()
print(f'Duplicados encontrados: {duplicates}')

df = df.drop_duplicates()

print(f'Registros después de eliminar duplicados: {len(df)}')


Duplicados encontrados: 22
Registros después de eliminar duplicados: 753


## 4. Estandarización de Variables Categóricas

In [32]:

df['gender'] = df['gender'].str.strip().str.upper()

gender_map = {
    'M':'MALE',
    'MALE':'MALE',
    'F':'FEMALE',
    'FEMALE':'FEMALE'
}

df['gender'] = df['gender'].replace(gender_map)

df['shift'] = df['shift'].str.strip().str.lower()

shift_map = {
    'day':'day',
    'day shift':'day',
    'night':'night',
    'night shift':'night'
}

df['shift'] = df['shift'].replace(shift_map)

print(df['gender'].value_counts())
print(df['shift'].value_counts())


gender
FEMALE    382
MALE      371
Name: count, dtype: int64
shift
day      379
night    374
Name: count, dtype: int64


## 5. Validación de Fechas

In [33]:

df['accident_date'] = pd.to_datetime(df['accident_date'])

future_records = df[df['accident_date'] > pd.Timestamp.today()]

print(f'Registros con fechas futuras: {len(future_records)}')
future_records.head()


Registros con fechas futuras: 8


,employee_id,age,gender,department,job_role,years_experience,shift,accident_date,accident_type,severity,lost_days,ppe_used,training_completed,fatigue_level,root_cause,city,overtime_hours
62,1014,43,FEMALE,Operations,Supervisor,19.0,day,2032-01-07,Fall,Minor,6,Yes,Yes,3.0,PPE Non-Compliance,Bogotá,13.0
136,1183,49,MALE,Maintenance,Operator,12.5,night,2032-01-08,Struck By Object,Moderate,2,No,No,3.0,Human Error,Cali,18.0
516,1068,43,FEMALE,Warehouse,Supervisor,23.9,day,2032-01-06,Fall,Minor,13,Yes,Yes,8.0,PPE Non-Compliance,Bogotá,11.5
531,1030,27,FEMALE,Quality,Supervisor,10.1,day,2032-01-04,Fall,Severe,9,Yes,Yes,9.0,Lack of Training,Medellín,20.5
588,1166,48,MALE,Logistics,Technician,20.5,day,2032-01-01,Fall,Minor,14,Yes,Yes,3.0,PPE Non-Compliance,Cali,0.0


In [34]:

df = df[df['accident_date'] <= pd.Timestamp.today()]


## 6. Validación de Reglas de Negocio

In [35]:

invalid_age = df[(df['age'] < 18) | (df['age'] > 70)]

print(f'Registros con edad inválida: {len(invalid_age)}')

invalid_age[['employee_id','age']].head()


Registros con edad inválida: 6


,employee_id,age
23,1051,120
168,1287,95
384,1244,15
406,1299,12
700,1054,14


In [36]:

df = df[(df['age'] >= 18) & (df['age'] <= 70)]


## 7. Detección de Outliers

In [38]:

def remove_outliers_iqr(dataframe, column):

    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)

    return dataframe[(dataframe[column] >= lower) &
                     (dataframe[column] <= upper)]

df = remove_outliers_iqr(df, 'lost_days')
df = remove_outliers_iqr(df, 'overtime_hours')

print(df.shape)


(721, 17)


## 8. Verificación Final

In [40]:

print(df.isnull().sum())
print(df.duplicated().sum())
print(df.shape)


employee_id           0
age                   0
gender                0
department            0
job_role              0
years_experience      0
shift                 0
accident_date         0
accident_type         0
severity              0
lost_days             0
ppe_used              0
training_completed    0
fatigue_level         0
root_cause            0
city                  0
overtime_hours        0
dtype: int64
2
(721, 17)


In [42]:
import os

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('notebooks', exist_ok=True)
os.makedirs('reports', exist_ok=True)
os.makedirs('images', exist_ok=True)

print("Estructura creada correctamente")

Estructura creada correctamente


## 9. Exportación del Dataset Limpio

In [43]:

output_path = 'data/cleaned/workplace_safety_dataset_cleaned.csv'

df.to_csv(output_path, index=False)

print("Dataset exportado correctamente")
print(output_path)


Dataset exportado correctamente
data/cleaned/workplace_safety_dataset_cleaned.csv



# Conclusiones

Durante el proceso de limpieza se realizaron las siguientes actividades:

- Imputación de valores nulos.
- Eliminación de duplicados.
- Estandarización de categorías.
- Eliminación de fechas futuras.
- Validación de reglas de negocio.
- Tratamiento de outliers.

El dataset resultante queda preparado para la fase de Análisis Exploratorio de Datos (EDA).
